In [48]:
import pathlib
from datetime import date, timedelta

import os
import sys
import re

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import jqdatasdk as jq
import tushare as ts

In [2]:
project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve() # Current Working Directory

for candidate_root in [current_path, * current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录")

from config.settings import settings

In [3]:
pro = ts.pro_api(settings.tushare_token)
jq.auth(settings.jqdata_id, settings.jqdata_secret)

auth success 


In [6]:
# =========================
# 2. 参数
# =========================

VARIETY = "RB"
START_DATE = "2026-01-01"
END_DATE = "2026-01-31"


In [8]:
# =========================
# 全时间轴全局检验：
# 是否存在同一天、同一品种、不同合约的 trade_time 规则不同
# 依赖 testing_04 语境：jq 已 auth；已 import pandas as pd 也没关系，这里会重复 import
# =========================

import re
from collections import defaultdict
import pandas as pd
from IPython.display import display

BATCH_SIZE = 200
ONLY_TRADE_DAYS = True   # True: 只把包含交易日的重叠区间视为有效冲突；False: 按自然日判断

exchange_name_map = {
    "XSGE": "上期所",
    "XDCE": "大商所",
    "XZCE": "郑商所",
    "CCFX": "中金所",
    "XINE": "上期能源",
    "GFEX": "广期所",
    "XGFE": "广期所",
}

def parse_contract(code):
    code = str(code).upper()
    m = re.match(r"^([A-Z]+)(\d+)\.(\w+)$", code)
    if m is None:
        return None, None, None
    return m.group(1), m.group(2), m.group(3)

def is_real_future_contract(code):
    variety, month, exchange = parse_contract(code)
    return variety is not None and month not in {"8888", "9999"}

def chunks(xs, n):
    for i in range(0, len(xs), n):
        yield xs[i:i + n]

def sessions_to_text(sessions):
    return " | ".join(sessions)

def normalize_day(x):
    return pd.to_datetime(x).normalize()

# 1. 全历史真实期货合约
all_fut = jq.get_all_securities(types=["futures"], date=None)

security_rows = []
for code, row in all_fut.iterrows():
    if not is_real_future_contract(code):
        continue

    variety, month, exchange = parse_contract(code)

    security_rows.append({
        "contract": str(code).upper(),
        "exchange": exchange,
        "exchange_name": exchange_name_map.get(exchange, exchange),
        "variety": variety,
        "contract_month": month,
        "security_start": normalize_day(row["start_date"]),
        "security_end": normalize_day(row["end_date"]),
        "display_name": row.get("display_name"),
    })

security_detail = pd.DataFrame(security_rows)

print(f"真实期货合约数: {len(security_detail)}")
print(f"品种数: {security_detail['variety'].nunique()}")

# 2. 批量获取 get_futures_info
info_cache = {}
failed_info_codes = []

codes = security_detail["contract"].tolist()

for batch in chunks(codes, BATCH_SIZE):
    try:
        info_cache.update(jq.get_futures_info(batch))
    except Exception as e:
        print("batch failed, fallback single:", batch[:5], "...", e)
        for code in batch:
            try:
                info_cache.update(jq.get_futures_info(code))
            except Exception as single_e:
                failed_info_codes.append({
                    "contract": code,
                    "error": str(single_e),
                })

print(f"成功获取 futures_info: {len(info_cache)}")
print(f"失败 futures_info: {len(failed_info_codes)}")

# 3. 合约-交易时间规则段：取合约上市区间与 trade_time 生效区间的交集
interval_rows = []
missing_rule_rows = []

for row in security_detail.itertuples(index=False):
    info = info_cache.get(row.contract)

    if not isinstance(info, dict):
        missing_rule_rows.append({
            "contract": row.contract,
            "reason": "no_futures_info",
        })
        continue

    trade_time = info.get("trade_time", [])

    if not trade_time:
        missing_rule_rows.append({
            "contract": row.contract,
            "reason": "empty_trade_time",
        })
        continue

    for rule in trade_time:
        if len(rule) < 3:
            missing_rule_rows.append({
                "contract": row.contract,
                "reason": f"invalid_trade_time_row: {rule}",
            })
            continue

        rule_start = normalize_day(rule[0])
        rule_end = normalize_day(rule[1])

        active_start = max(row.security_start, rule_start)
        active_end = min(row.security_end, rule_end)

        if active_start > active_end:
            continue

        sessions_key = tuple(rule[2:])
        sessions_text = sessions_to_text(sessions_key)

        interval_rows.append({
            "contract": row.contract,
            "exchange": row.exchange,
            "exchange_name": row.exchange_name,
            "variety": row.variety,
            "contract_month": row.contract_month,
            "active_start": active_start,
            "active_end": active_end,
            "security_start": row.security_start,
            "security_end": row.security_end,
            "rule_start": rule_start,
            "rule_end": rule_end,
            "sessions_key": sessions_key,
            "sessions_text": sessions_text,
        })

rule_interval_detail = pd.DataFrame(interval_rows)
missing_rule_detail = pd.DataFrame(missing_rule_rows)

if rule_interval_detail.empty:
    raise ValueError("rule_interval_detail 为空，无法检验。")

print(f"合约-规则有效区间数: {len(rule_interval_detail)}")

# 4. 交易日过滤工具
if ONLY_TRADE_DAYS:
    all_trade_days = pd.DatetimeIndex(
        pd.to_datetime(jq.get_all_trade_days()).normalize()
    ).sort_values()

    def trade_day_window(start, end):
        left = all_trade_days.searchsorted(start)
        right = all_trade_days.searchsorted(end, side="right")

        if right <= left:
            return None

        return {
            "first_trade_day": all_trade_days[left],
            "last_trade_day": all_trade_days[right - 1],
            "n_trade_days": int(right - left),
        }
else:
    def trade_day_window(start, end):
        return {
            "first_trade_day": start,
            "last_trade_day": end,
            "n_trade_days": int((end - start).days) + 1,
        }

# 5. 区间扫描：同一 exchange + variety 内，寻找不同 sessions 的重叠区间
conflict_rows = []

for (exchange, variety), g in rule_interval_detail.groupby(["exchange", "variety"]):
    g = g.reset_index(drop=True)

    events = defaultdict(list)

    for i, r in g.iterrows():
        events[r["active_start"]].append(("add", i))
        events[r["active_end"] + pd.Timedelta(days=1)].append(("remove", i))

    dates = sorted(events.keys())
    active = {}

    for pos, d in enumerate(dates):
        for op, i in events[d]:
            if op == "remove":
                active.pop(i, None)

        for op, i in events[d]:
            if op == "add":
                active[i] = g.loc[i].to_dict()

        if pos + 1 >= len(dates):
            continue

        period_start = d
        period_end = dates[pos + 1] - pd.Timedelta(days=1)

        if period_start > period_end or not active:
            continue

        tdw = trade_day_window(period_start, period_end)
        if tdw is None:
            continue

        by_rule = defaultdict(list)

        for item in active.values():
            by_rule[item["sessions_text"]].append(item["contract"])

        if len(by_rule) <= 1:
            continue

        rule_contracts = {
            sessions_text: sorted(set(contracts))
            for sessions_text, contracts in by_rule.items()
        }

        all_contracts = sorted({
            contract
            for contracts in rule_contracts.values()
            for contract in contracts
        })

        conflict_rows.append({
            "exchange": exchange,
            "exchange_name": exchange_name_map.get(exchange, exchange),
            "variety": variety,
            "period_start": period_start,
            "period_end": period_end,
            "first_trade_day": tdw["first_trade_day"],
            "last_trade_day": tdw["last_trade_day"],
            "n_trade_days": tdw["n_trade_days"],
            "n_contracts": len(all_contracts),
            "n_session_rules": len(rule_contracts),
            "contracts": all_contracts,
            "rule_contracts": rule_contracts,
        })

global_conflict_periods = pd.DataFrame(conflict_rows)

if global_conflict_periods.empty:
    print("结论：未发现同一天、同一品种、不同合约交易时间规则不同的情况。")
else:
    global_conflict_periods = global_conflict_periods.sort_values(
        ["exchange", "variety", "first_trade_day", "last_trade_day"]
    ).reset_index(drop=True)

    print(f"结论：发现 {len(global_conflict_periods)} 个冲突区间。")
    display(global_conflict_periods)

# 6. 品种级汇总
if global_conflict_periods.empty:
    global_conflict_summary = pd.DataFrame()
else:
    global_conflict_summary = (
        global_conflict_periods
        .groupby(["exchange", "exchange_name", "variety"], as_index=False)
        .agg(
            n_conflict_periods=("period_start", "count"),
            first_conflict_trade_day=("first_trade_day", "min"),
            last_conflict_trade_day=("last_trade_day", "max"),
            max_session_rules_same_day=("n_session_rules", "max"),
            max_contracts_same_day=("n_contracts", "max"),
        )
        .sort_values(["exchange", "variety"])
        .reset_index(drop=True)
    )

    print("品种级冲突汇总：")
    display(global_conflict_summary)

print("输出对象：")
print("security_detail          # 全历史真实合约清单")
print("rule_interval_detail     # 合约-交易时间规则有效区间")
print("global_conflict_periods  # 全局冲突区间")
print("global_conflict_summary  # 品种级冲突汇总")
print("missing_rule_detail      # futures_info/trade_time 缺失或异常记录")

E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)


真实期货合约数: 10389
品种数: 95
成功获取 futures_info: 10389
失败 futures_info: 0
合约-规则有效区间数: 11026


E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)


结论：未发现同一天、同一品种、不同合约交易时间规则不同的情况。
输出对象：
security_detail          # 全历史真实合约清单
rule_interval_detail     # 合约-交易时间规则有效区间
global_conflict_periods  # 全局冲突区间
global_conflict_summary  # 品种级冲突汇总
missing_rule_detail      # futures_info/trade_time 缺失或异常记录


In [15]:
jq.get_future_contracts("RB", "2026-07-08")

['RB2607.XSGE',
 'RB2608.XSGE',
 'RB2609.XSGE',
 'RB2610.XSGE',
 'RB2611.XSGE',
 'RB2612.XSGE',
 'RB2701.XSGE',
 'RB2702.XSGE',
 'RB2703.XSGE',
 'RB2704.XSGE',
 'RB2705.XSGE',
 'RB2706.XSGE']

In [ ]:
jq.get_price(["RB2610.XSGE"], start_date="2026-07-07", end_date="2026-07-08", frequency="1m")

In [ ]:
jq.get_price(["CU2608.XSGE"], start_date="2026-07-07", end_date="2026-07-08", frequency="1m")

In [ ]:
jq.get_price(["AU2612.XSGE"], start_date="2026-07-07 21:00:00", end_date="2026-07-08", frequency="1m")

In [24]:
import pandas as pd

def audit_daily_boundary(jq, code, trading_day, windows):
    daily = jq.get_price(
        code,
        start_date=trading_day,
        end_date=trading_day,
        frequency="1d",
        fields=["open", "close", "high", "low", "volume"],
        fq=None,
    )

    minute = jq.get_price(
        code,
        start_date=windows[0][0],
        end_date=windows[-1][1],
        frequency="1m",
        fields=["open", "close", "high", "low", "volume"],
        fq=None,
    )

    minute = minute.reset_index().rename(columns={"index": "time"})
    minute["time"] = pd.to_datetime(minute["time"])

    mask = False
    for start, end in windows:
        mask = mask | minute["time"].between(
            pd.Timestamp(start),
            pd.Timestamp(end),
            inclusive="both",
        )

    m = minute.loc[mask].dropna(subset=["open", "close"])
    first = m.iloc[0]
    last = m.iloc[-1]
    d = daily.iloc[0]

    return pd.DataFrame([{
        "code": code,
        "trading_day": trading_day,
        "daily_open": d["open"],
        "first_minute": first["time"],
        "first_minute_open": first["open"],
        "open_match": abs(d["open"] - first["open"]) < 1e-8,
        "daily_close": d["close"],
        "last_minute": last["time"],
        "last_minute_close": last["close"],
        "close_match": abs(d["close"] - last["close"]) < 1e-8,
    }])

windows = [
    ("2026-07-07 21:00:00", "2026-07-08 01:00:00"),
    ("2026-07-08 09:00:00", "2026-07-08 10:15:00"),
    ("2026-07-08 10:30:00", "2026-07-08 11:30:00"),
    ("2026-07-08 13:30:00", "2026-07-08 15:00:00"),
]

print(audit_daily_boundary(jq, "CU2608.XSGE", "2026-07-08", windows))

E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)


          code trading_day  daily_open        first_minute  first_minute_open  open_match  daily_close         last_minute  last_minute_close  close_match
0  CU2608.XSGE  2026-07-08    103080.0 2026-07-07 21:01:00           103080.0        True     102850.0 2026-07-08 15:00:00           102850.0         True


E:\anaconda3\envs\latitude\Lib\site-packages\jqdatasdk\compat\pickle_compat.py:28: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  return pickle.load(fh, encoding=encoding)


In [ ]:
jq.get_all_securities(types=['futures'],date='2026-07-05')

In [29]:
jq.get_future_contracts('2017-01-05')

[]

In [72]:
current_date = "2026-07-06"

all_securities_daily_df = jq.get_all_securities(types = ['futures'], date = current_date)
# all_securities_daily_df

In [73]:
pattern = re.compile(r"^([A-Z]+)\d{4}\.([A-Z]+)$")

contract_series = (
    all_securities_daily_df.index
    .to_series(name="contract")
    .drop_duplicates()
    .reset_index(drop=True)
)

future_exchange_df = (
    contract_series
    .str.extract(pattern, expand=True)
    .rename(columns={0: "future", 1: "exchange"})

    .assign(contract=contract_series)
    .loc[:, ["contract", "future", "exchange"]]
    .loc[lambda df: df["exchange"].ne("CCFX")]

    .drop_duplicates(ignore_index=True)
    .dropna()
)

display(future_exchange_df.head())
display(future_exchange_df.tail())

,contract,future,exchange
0,A2607.XDCE,A,XDCE
1,A2609.XDCE,A,XDCE
2,A2611.XDCE,A,XDCE
3,A2701.XDCE,A,XDCE
4,A2703.XDCE,A,XDCE


,contract,future,exchange
978,ZN2704.XSGE,ZN,XSGE
979,ZN2705.XSGE,ZN,XSGE
980,ZN2706.XSGE,ZN,XSGE
981,ZN8888.XSGE,ZN,XSGE
982,ZN9999.XSGE,ZN,XSGE


In [80]:
# jq.get_price(
#     security   = future_exchange_df["contract"].to_list(),
#     start_date = current_date,
#     end_date   = current_date,
#     frequency  = 'daily',
#     fields     = ['volume','money']
# )

In [101]:
def is_trade_date(trade_date) -> bool:
    trade_days = jq.get_trade_days(start_date = trade_date, end_date = trade_date)
    return len(trade_days) == 1

In [104]:
is_trade_date("2026-07-07")

True